# SecureSpeak — Phase 1: Dataset Audit

## Purpose
This notebook does **ONE thing**: it reads every dataset you currently use and produces an honest health report for each. **No training. No model changes. No new datasets.** Just an x-ray of what you have right now.

**Why this matters:** Before we add new datasets or change the pipeline, we need to know what's actually in the data we already have. Earlier we found a worrying line in your old notebook: `StealthPhisher sampled: Phishing: 0` — meaning a label parsing bug at some point. This audit catches issues like that **before** they propagate into a published paper.

## Folder Layout (set up once, used forever)
```
cse498R/
├── Datasets/                       ← read-only inputs (you already have this)
│   ├── StealthPhisher2025.csv
│   ├── BangalaBarta bangla_spam_sms smishing.csv
│   ├── CICMalAnal2017/             ← folder with Benign-CSVs/, Adware-CSVs/, etc.
│   ├── pcapdroid_merge.csv
│   ├── PCAPdroid_zeroday_test_cleandata.csv
│   ├── UNSW_NB15_training-set.csv
│   └── UNSW_NB15_testing-set.csv
│
└── model_for_research/             ← EVERYTHING this notebook creates goes here
    └── phase1_audit/
        ├── reports/                ← one .txt health report per dataset
        ├── figures/                ← class-balance plots, NaN heatmaps
        └── audit_summary.json      ← machine-readable summary for later phases
```

## How to use this notebook
1. Mount Drive (Step 0).
2. Verify all dataset paths (Step 1).
3. Run each audit cell in order (Steps 2–7).
4. Read `audit_summary.json` at the end. That's your verdict.

**No cell trains anything. No cell modifies your existing data.** Safe to run as many times as you want.

---
## Step 0 — Environment + Drive Mount
**Produces:** `BASE_DATA`, `BASE_OUT`, all subfolder paths.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, glob, hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── PATHS — CHANGE NOTHING BELOW THIS UNLESS YOUR FOLDER NAMES DIFFER ────────
BASE_DATA = '/content/drive/MyDrive/cse498R/Datasets'
BASE_OUT  = '/content/drive/MyDrive/cse498R/model_for_research'

# Phase 1 output structure
PHASE1_ROOT     = os.path.join(BASE_OUT, 'phase1_audit')
REPORTS_DIR     = os.path.join(PHASE1_ROOT, 'reports')
FIGURES_DIR     = os.path.join(PHASE1_ROOT, 'figures')
SUMMARY_JSON    = os.path.join(PHASE1_ROOT, 'audit_summary.json')

for d in [BASE_OUT, PHASE1_ROOT, REPORTS_DIR, FIGURES_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

# Dataset paths (match your existing v5 notebook)
STEALTH_CSV     = os.path.join(BASE_DATA, 'StealthPhisher2025.csv')
BANGLA_CSV      = os.path.join(BASE_DATA, 'BangalaBarta bangla_spam_sms smishing.csv')
UNSW_TR         = os.path.join(BASE_DATA, 'UNSW_NB15_training-set.csv')
UNSW_TE         = os.path.join(BASE_DATA, 'UNSW_NB15_testing-set.csv')
PCAP_REAL       = os.path.join(BASE_DATA, 'pcapdroid_merge.csv')
PCAP_BLIND      = os.path.join(BASE_DATA, 'PCAPdroid_zeroday_test_cleandata.csv')
CICMAL_DIR      = None  # auto-detected below — folder name varies

# Locate CICMalAnal2017 folder (your notebook does this dynamically)
for d in os.listdir(BASE_DATA):
    if 'CICMalAnal' in d and os.path.isdir(os.path.join(BASE_DATA, d)):
        CICMAL_DIR = os.path.join(BASE_DATA, d)
        break

print('═' * 70)
print(' SECURESPEAK PHASE 1 AUDIT — PATHS')
print('═' * 70)
print(f'Input  (Datasets):           {BASE_DATA}')
print(f'Output (model_for_research): {BASE_OUT}')
print(f'Reports → {REPORTS_DIR}')
print(f'Figures → {FIGURES_DIR}')
print(f'Summary → {SUMMARY_JSON}')
print()
print('Today:', datetime.now().strftime('%Y-%m-%d %H:%M'))

---
## Step 1 — Verify Every File Exists Before Auditing
If a file is missing, this cell tells you immediately so you don't waste time.

In [ ]:
files_to_check = [
    ('StealthPhisher 2025',  STEALTH_CSV,  True),
    ('BangalaBarta SMS',     BANGLA_CSV,   True),
    ('UNSW-NB15 train',      UNSW_TR,      False),  # optional, may be dropped later
    ('UNSW-NB15 test',       UNSW_TE,      False),
    ('PCAPdroid real',       PCAP_REAL,    True),
    ('PCAPdroid blind',      PCAP_BLIND,   False),
    ('CICMalAnal2017 dir',   CICMAL_DIR,   True),
]

print(f'{"DATASET":<24} {"STATUS":<10} {"SIZE":<12} PATH')
print('─' * 100)
missing_required = []
for label, path, required in files_to_check:
    if path is None:
        status, size_str = 'MISSING', '—'
    elif os.path.isdir(path):
        # directory size = sum of immediate file sizes
        total = sum(os.path.getsize(os.path.join(r, f))
                    for r, _, fs in os.walk(path) for f in fs)
        status, size_str = 'OK (dir)', f'{total/1024**2:.1f} MB'
    elif os.path.exists(path):
        status = 'OK'
        size_str = f'{os.path.getsize(path)/1024**2:.1f} MB'
    else:
        status, size_str = 'MISSING', '—'
        if required:
            missing_required.append(label)
    flag = ' [opt]' if not required else ''
    print(f'  {label:<22} {status:<10} {size_str:<12} {path}{flag}')

print()
if missing_required:
    print('❌ STOP — these REQUIRED files are missing:')
    for m in missing_required:
        print(f'     - {m}')
    print('\nFix paths above or upload missing files before continuing.')
else:
    print('✓ All required files present. Proceeding to audits.')

---
## Step 2 — Helper Functions for Auditing
Reusable utilities. Each audit step uses these to produce a consistent report format.

**No need to read the code carefully** — these are just shared helpers. Just run the cell.

In [ ]:
def write_report(dataset_name: str, lines: list):
    """Save a per-dataset health report as plain text."""
    safe_name = dataset_name.lower().replace(' ', '_').replace('/', '_')
    path = os.path.join(REPORTS_DIR, f'{safe_name}.txt')
    header = [
        '═' * 70,
        f' DATASET HEALTH REPORT — {dataset_name}',
        f' Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
        '═' * 70,
        '',
    ]
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(header + lines))
    print(f'  → Report saved: {path}')
    return path

def basic_df_stats(df: pd.DataFrame, sample_for_dups: int = 500_000) -> dict:
    """Return dict of basic statistics for a DataFrame."""
    n_rows, n_cols = df.shape
    nan_per_col = df.isna().sum()
    inf_count = 0
    try:
        numeric = df.select_dtypes(include=[np.number])
        inf_count = int(np.isinf(numeric.values).sum()) if numeric.shape[1] else 0
    except Exception:
        pass
    # Duplicate check on a sample (full check on huge files is slow)
    if n_rows > sample_for_dups:
        dup_sample = df.sample(sample_for_dups, random_state=42)
        dup_count_in_sample = int(dup_sample.duplicated().sum())
        dup_note = f'{dup_count_in_sample} duplicates in {sample_for_dups}-row sample'
    else:
        dup_count = int(df.duplicated().sum())
        dup_note = f'{dup_count} duplicates (full check)'
    return {
        'n_rows': int(n_rows),
        'n_cols': int(n_cols),
        'total_nan': int(nan_per_col.sum()),
        'cols_with_nan': int((nan_per_col > 0).sum()),
        'inf_count': inf_count,
        'dup_note': dup_note,
        'dtypes_summary': df.dtypes.astype(str).value_counts().to_dict(),
    }

def plot_class_balance(values, title, save_path):
    """Save a horizontal bar chart of class counts."""
    counts = pd.Series(values).value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(7, max(2.5, 0.4 * len(counts))))
    counts.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(v, i, f' {v:,}', va='center')
    fig.tight_layout()
    fig.savefig(save_path, dpi=110, bbox_inches='tight')
    plt.close(fig)
    print(f'  → Figure saved: {save_path}')

# Master summary dict — every audit appends its verdict here
AUDIT_SUMMARY = {
    'generated_at': datetime.now().isoformat(),
    'datasets': {},
}

print('✓ Helpers loaded. Ready for per-dataset audits.')

---
## Step 3 — Audit: StealthPhisher 2025 (URL classifier source)

**What we check:**
- File loads without parsing errors
- Label column is found and parsed correctly (fixes the historical "Phishing: 0" issue)
- Class balance (legit vs phishing) — should be roughly balanced
- NaN, infinite, duplicate rows
- URL column has actual URLs (not garbage)

**Expected:** ~3.3M rows, balanced classes, URL column present.

In [ ]:
lines = []
ok = True

print('Reading StealthPhisher 2025 (this may take 30–60 seconds)...')
df_sp = pd.read_csv(STEALTH_CSV, low_memory=False)
stats = basic_df_stats(df_sp)

lines.append(f'Rows: {stats["n_rows"]:,}')
lines.append(f'Columns: {stats["n_cols"]}')
lines.append(f'Column names (first 15): {df_sp.columns.tolist()[:15]}')
lines.append(f'Total NaN cells: {stats["total_nan"]:,}')
lines.append(f'Columns containing NaN: {stats["cols_with_nan"]}')
lines.append(f'Infinite numeric values: {stats["inf_count"]}')
lines.append(f'Duplicates: {stats["dup_note"]}')
lines.append(f'Dtype mix: {stats["dtypes_summary"]}')
lines.append('')

# Label parsing — this is the historically broken cell
label_col_candidates = ['Label', 'label', 'class', 'target', 'is_phishing']
label_col = next((c for c in label_col_candidates if c in df_sp.columns), None)
if label_col is None:
    lines.append('❌ NO LABEL COLUMN FOUND — investigate before using this dataset.')
    ok = False
else:
    lines.append(f'Label column found: "{label_col}"')
    raw_counts = df_sp[label_col].value_counts(dropna=False).to_dict()
    lines.append(f'Raw label distribution: {raw_counts}')
    # Sanity: do we have both classes?
    n_classes = df_sp[label_col].nunique(dropna=True)
    lines.append(f'Unique label values: {n_classes}')
    if n_classes < 2:
        lines.append('❌ CRITICAL — only one class present. Label parsing is broken.')
        ok = False
    else:
        lines.append('✓ Both classes present.')

# URL column sanity
url_col = next((c for c in ['URL', 'url', 'Url', 'website'] if c in df_sp.columns), None)
if url_col:
    sample_urls = df_sp[url_col].dropna().head(3).tolist()
    lines.append(f'URL column: "{url_col}"')
    lines.append(f'Sample URLs: {sample_urls}')
    looks_like_url = df_sp[url_col].astype(str).str.contains(r'http|www|\.', regex=True).mean()
    lines.append(f'Fraction looking like URLs: {looks_like_url:.2%}')
    if looks_like_url < 0.5:
        lines.append('⚠ Warning: URL column may not contain real URLs.')
else:
    lines.append('⚠ No URL column found — verify column names.')

# Plot class balance
if label_col:
    plot_class_balance(
        df_sp[label_col].astype(str),
        'StealthPhisher 2025 — class balance',
        os.path.join(FIGURES_DIR, 'stealth_class_balance.png'),
    )

# Verdict
lines.append('')
lines.append('VERDICT: ' + ('✓ HEALTHY — safe to use as-is.' if ok else '❌ NEEDS FIX before training.'))
write_report('StealthPhisher 2025', lines)
AUDIT_SUMMARY['datasets']['stealthphisher_2025'] = {
    'rows': stats['n_rows'], 'cols': stats['n_cols'],
    'label_col': label_col, 'verdict': 'healthy' if ok else 'needs_fix',
}
print('\n'.join(lines[-6:]))
del df_sp  # free memory

---
## Step 4 — Audit: BangalaBarta SMS (Bangla smishing source)

**What we check:**
- File loads with UTF-8 (Bangla characters preserved)
- Label column present
- **Critical:** the smish/promo merging issue we discussed — are these labels currently merged into one class?
- Sample message inspection (Bangla text rendering OK?)

**Expected:** 2,772 messages, multiple label categories.

In [ ]:
lines = []
ok = True

df_bn = pd.read_csv(BANGLA_CSV, encoding='utf-8')
df_bn.columns = df_bn.columns.str.strip().str.lower()
stats = basic_df_stats(df_bn)

lines.append(f'Rows: {stats["n_rows"]:,}')
lines.append(f'Columns: {df_bn.columns.tolist()}')
lines.append(f'Total NaN: {stats["total_nan"]}')
lines.append(f'Duplicates: {stats["dup_note"]}')
lines.append('')

text_col  = next((c for c in ['message','text','sms','content','msg'] if c in df_bn.columns), df_bn.columns[0])
label_col = next((c for c in ['label','class','spam','type','category'] if c in df_bn.columns), None)

lines.append(f'Text column:  "{text_col}"')
lines.append(f'Label column: "{label_col}"')
lines.append('')

# Show raw label distribution BEFORE any merging
if label_col:
    raw_counts = df_bn[label_col].astype(str).str.lower().value_counts().to_dict()
    lines.append('Raw labels (BEFORE smish/promo merge):')
    for k, v in raw_counts.items():
        lines.append(f'    {k:<15} {v}')
    n_classes = len(raw_counts)
    lines.append('')
    if n_classes >= 3:
        lines.append('⚠ MULTIPLE label categories detected. The v5 notebook MERGES some of these')
        lines.append('  (smish + promo + spam → 1, else 0). This inflates the reported accuracy')
        lines.append('  by mixing promotional spam (easy) with real smishing (hard).')
        lines.append('  RECOMMENDATION: For the next training run, evaluate two ways:')
        lines.append('    (a) smish-vs-normal only (drop promo) — the honest smishing task')
        lines.append('    (b) 3-class (smish/promo/normal) — report per-class metrics')

# Sample messages (Bangla rendering check)
lines.append('')
lines.append('Sample messages (verify Bangla characters look correct):')
for i, msg in enumerate(df_bn[text_col].dropna().head(3).tolist(), 1):
    snippet = str(msg)[:120].replace('\n', ' ')
    lines.append(f'    [{i}] {snippet}{"..." if len(str(msg)) > 120 else ""}')

# Plot class balance
if label_col:
    plot_class_balance(
        df_bn[label_col].astype(str).str.lower(),
        'BangalaBarta SMS — raw class balance (before merging)',
        os.path.join(FIGURES_DIR, 'bangla_class_balance.png'),
    )

lines.append('')
lines.append('VERDICT: ✓ Data is usable. Action required: in Phase 2, re-train SMS')
lines.append('         classifier with promo split out from smish for honest metrics.')
write_report('BangalaBarta SMS', lines)
AUDIT_SUMMARY['datasets']['bangalabarta_sms'] = {
    'rows': stats['n_rows'], 'cols': stats['n_cols'],
    'label_col': label_col, 'verdict': 'usable_needs_relabel',
}
print('\n'.join(lines))
del df_bn

---
## Step 5 — Audit: CICMalAnal2017 (Network malware source)

**What we check:**
- All 5 family folders exist (Benign, SMS, Adware, Ransomware, Scareware)
- Each folder has CSV files with consistent column counts
- **Critical:** SCAREWARE family is physically separable (the zero-day holdout claim)
- Column names match across families (no silent feature drift)

**Expected:** 5 family folders, ~80,000 flows each, ~85 CICFlowMeter columns.

In [ ]:
lines = []
ok = True

FAM_MAP = {
    'Benign-CSVs':      ('BENIGN',    'NONE'),
    'SMSmalware-CSVs':  ('MALICIOUS', 'SMS_MALWARE'),
    'Adware-CSVs':      ('MALICIOUS', 'ADWARE'),
    'Ransomware-CSVs':  ('MALICIOUS', 'RANSOMWARE'),
    'Scareware-CSVs':   ('MALICIOUS', 'SCAREWARE'),
}

lines.append(f'CICMalAnal2017 root: {CICMAL_DIR}')
lines.append('')

family_summary = []
ref_cols = None
col_drift = False

for folder, (binlbl, family) in FAM_MAP.items():
    cat_path = os.path.join(CICMAL_DIR, folder)
    if not os.path.isdir(cat_path):
        lines.append(f'❌ MISSING folder: {folder}')
        family_summary.append((family, 'MISSING', 0, 0))
        ok = False
        continue

    csvs = sorted(glob.glob(os.path.join(cat_path, '**', '*.csv'), recursive=True))
    if not csvs:
        lines.append(f'⚠ Folder {folder} exists but contains no CSV files.')
        family_summary.append((family, 'EMPTY', 0, 0))
        continue

    # Read just the first CSV to check column structure
    first = pd.read_csv(csvs[0], nrows=5, low_memory=False)
    n_cols = len(first.columns)

    # Compare columns across families
    cols_set = set(first.columns)
    if ref_cols is None:
        ref_cols = cols_set
        ref_family = family
    elif cols_set != ref_cols:
        diff = cols_set.symmetric_difference(ref_cols)
        lines.append(f'⚠ Column drift in {family}: {len(diff)} differing columns vs {ref_family}')
        col_drift = True

    # Estimate row count cheaply (count newlines in first file, scale)
    n_rows_est = sum(1 for _ in open(csvs[0])) - 1
    total_est = n_rows_est * len(csvs)
    family_summary.append((family, 'OK', len(csvs), total_est))

lines.append(f'{"FAMILY":<14} {"STATUS":<8} {"FILES":>6} {"~ROWS (estimate)":>20}')
lines.append('─' * 55)
for fam, status, n_files, n_rows in family_summary:
    lines.append(f'{fam:<14} {status:<8} {n_files:>6} {n_rows:>20,}')
lines.append('')

if col_drift:
    lines.append('⚠ Column drift detected across families. Verify CICFlowMeter version')
    lines.append('  consistency before training. Use the intersection of columns.')
else:
    lines.append('✓ Column schema consistent across all family folders.')

# Verify SCAREWARE folder is separately addressable (the zero-day claim)
scareware_path = os.path.join(CICMAL_DIR, 'Scareware-CSVs')
if os.path.isdir(scareware_path):
    lines.append(f'✓ SCAREWARE folder is physically separate at: {scareware_path}')
    lines.append('  → Zero-day holdout is structurally enforceable.')
else:
    lines.append('❌ SCAREWARE folder not found — zero-day evaluation cannot proceed.')
    ok = False

lines.append('')
lines.append('VERDICT: ' + ('✓ HEALTHY' if ok else '❌ NEEDS FIX'))
lines.append('NOTE: 2017 is old. Phase 5 of the plan adds CICMalDroid2020 syscall')
lines.append('      data as a complementary experiment (different feature space).')

write_report('CICMalAnal2017', lines)
AUDIT_SUMMARY['datasets']['cicmalanal_2017'] = {
    'families': [{'name': f, 'status': s, 'files': nf, 'est_rows': nr}
                 for f, s, nf, nr in family_summary],
    'column_drift': col_drift,
    'verdict': 'healthy' if ok else 'needs_fix',
}
print('\n'.join(lines))

---
## Step 6 — Audit: PCAPdroid Bangladesh (your best asset, 52K real flows)

**What we check:**
- Row count matches the 52,118 claim in the paper
- Columns align with CICFlowMeter (so you can score them with the network model)
- Anonymization done (no real IPs in the file)
- Per-app distribution (which apps dominate the capture)

In [ ]:
lines = []
ok = True

df_pcap = pd.read_csv(PCAP_REAL, low_memory=False)
stats = basic_df_stats(df_pcap)

lines.append(f'Rows: {stats["n_rows"]:,}  (paper claims 52,118)')
lines.append(f'Columns: {stats["n_cols"]}')
lines.append(f'Column names: {df_pcap.columns.tolist()}')
lines.append(f'NaN cells: {stats["total_nan"]:,}  (columns with NaN: {stats["cols_with_nan"]})')
lines.append(f'Infinite values: {stats["inf_count"]}')
lines.append(f'Duplicates: {stats["dup_note"]}')
lines.append('')

# Sanity: does row count match the paper?
expected = 52_118
if abs(stats['n_rows'] - expected) > 100:
    lines.append(f'⚠ Row count {stats["n_rows"]:,} differs from paper\'s 52,118 by more than 100.')
    lines.append('  Check whether this is a different version of the merge.')

# Anonymization check — search for any column that looks like an IP
ip_cols = [c for c in df_pcap.columns if 'ip' in c.lower() or 'addr' in c.lower()]
if ip_cols:
    lines.append(f'IP-like columns found: {ip_cols}')
    for col in ip_cols[:2]:
        sample = df_pcap[col].dropna().astype(str).head(3).tolist()
        lines.append(f'  Sample {col}: {sample}')
    lines.append('  → Verify these are hashed/anonymized, not raw IPs.')
else:
    lines.append('✓ No IP-named columns — anonymization likely OK.')

# Per-app distribution if an app column exists
app_col = next((c for c in df_pcap.columns if 'app' in c.lower() or 'package' in c.lower()), None)
if app_col:
    top_apps = df_pcap[app_col].value_counts().head(10).to_dict()
    lines.append('')
    lines.append(f'Top 10 apps by flow count ({app_col}):')
    for app, n in top_apps.items():
        lines.append(f'    {str(app)[:40]:<42} {n:,}')

lines.append('')
lines.append('VERDICT: ✓ This is your strongest asset. Real, novel, Bangladesh-specific.')
lines.append('         Phase 2 will use this as the base for real-attack injection.')

write_report('PCAPdroid Bangladesh', lines)
AUDIT_SUMMARY['datasets']['pcapdroid_bangladesh'] = {
    'rows': stats['n_rows'], 'cols': stats['n_cols'],
    'verdict': 'healthy_and_critical',
}
print('\n'.join(lines))
del df_pcap

---
## Step 7 — Audit: UNSW-NB15 (slated for removal)

Your paper currently includes UNSW-NB15 as a "cross-domain check" but only 8 of 22 features mapped — this hurts credibility. We audit it briefly here, then **the plan is to drop it from the paper.**

If the file isn't present, this step is skipped. No harm done.

In [ ]:
if os.path.exists(UNSW_TR):
    lines = []
    df_un = pd.read_csv(UNSW_TR, low_memory=False)
    stats = basic_df_stats(df_un)
    lines.append(f'Rows: {stats["n_rows"]:,}  Columns: {stats["n_cols"]}')
    lines.append(f'First 10 columns: {df_un.columns.tolist()[:10]}')
    lines.append('')
    lines.append('RECOMMENDATION: Drop UNSW-NB15 from the paper.')
    lines.append('Reason: 8/22 features mapped to CICFlowMeter — the result is not interpretable.')
    lines.append('Phase 6 of the plan removes this entirely.')
    write_report('UNSW-NB15', lines)
    AUDIT_SUMMARY['datasets']['unsw_nb15'] = {
        'rows': stats['n_rows'], 'cols': stats['n_cols'], 'verdict': 'drop_from_paper',
    }
    print('\n'.join(lines))
    del df_un
else:
    print('UNSW-NB15 not found — skipping. (This is fine; we plan to drop it anyway.)')

---
## Step 8 — Save Master Audit Summary

Writes a machine-readable JSON summary of all audits. Phase 2 will read this to know which datasets are ready vs. which need fixes.

In [ ]:
with open(SUMMARY_JSON, 'w', encoding='utf-8') as f:
    json.dump(AUDIT_SUMMARY, f, indent=2, ensure_ascii=False)

print('═' * 70)
print(' PHASE 1 AUDIT COMPLETE')
print('═' * 70)
print(f'\nSummary file: {SUMMARY_JSON}\n')
print('Datasets audited:')
for k, v in AUDIT_SUMMARY['datasets'].items():
    print(f'    {k:<30} → {v.get("verdict", "unknown")}')
print(f'\nText reports per dataset:  {REPORTS_DIR}')
print(f'Figures (class balances):  {FIGURES_DIR}')
print('\n✓ Phase 1 is done. Next: Phase 2 — replace synthetic fusion evaluation with real attacks.')

---
## What This Audit Did NOT Do (Intentionally)

- **No training.** We did not load or modify any model. Your v5 results are untouched.
- **No data modification.** Every CSV in `Datasets/` is read-only. We only opened, summarized, and closed.
- **No fusion-layer changes.** ECAFN, ReliabilityMLP, CGF — none of them were touched. That's Phase 2/3 work.
- **No new datasets downloaded.** PhishTank, PhiUSIIL, CICMalDroid2020 — those come in Phase 2.

## What to Do With the Output

1. Open every `.txt` file in `model_for_research/phase1_audit/reports/`.
2. Read the **VERDICT** line at the bottom of each.
3. Paste the full output of this notebook (or share the JSON summary) so we can plan Phase 2 grounded in what's actually true about your data — not what you think is true.

## Next Step

**Phase 2, Item 2.1 — Add PhishTank as external URL validation.** Once you confirm Phase 1 is healthy, we build the next notebook: `SecureSpeak_Phase2_ExternalValidation.ipynb`. Same structure: reads from `Datasets/`, writes to `model_for_research/phase2_external/`.